## Create the traffic simulator class

Here we will be running a simulation of a click / no click that a user on a site may go through. For this we will first start by defining the traffic simulator class which stores the simulated CTR rates calculated in the previous Python notebook.

In [ ]:
import numpy as np
class TrafficSimulator:

    def __init__(self, true_ctrs, seed=42):

        # store the simulated ctr rates of each variant in dictionary
        self.true_ctrs = true_ctrs

        # for reproducability create rng
        self.rng = np.random.default_rng(seed)

    def simulate_click(self, variant : str) -> int:

        # if not a valid variant return
        if variant not in self.true_ctrs:
            print('This variant does not exist.')
            return -1
        
        # get the random number 
        random_no = self.rng.random()

        # returns 1 is lower than p else 0
        return int(random_no < self.true_ctrs[variant])

    def simulate_batch(self, variant: str, n: int) -> int:

        if variant not in self.true_ctrs:
            print('This variant does not exist.')
            return -1

        # return result
        return (self.rng.random(n) < self.true_ctrs[variant]).astype(int)


In [2]:
true_ctrs = {'A' : 0.657, 'B': 0.2120, 'C' : 0.2167}

## A/B Testing

Now, we will run three different types of A/B testing

### Classic A/B testing


In classic A/B testing, we first need to determine the number of users that we should run our sampling. This can be done using a power test which tells us at which sample number will our experiments haev statistical significance. We have set the minimum detectable effect at 10% meaning we want to see if the difference between variance is atleast 10% with a baseline ctr of 0.21.

Then using the sample size, we will run our simulation on the three variants and use the chi-squared test to determine whether the differences in click rates are significant or could be from random noise. This test will tell us only if atleast one of the three are significantly different. But to pin-point which pairs of the variants differ, we conduct z-tests across all three pairs, using a Bonferroni correction since the number of false positives risk increase when conducting individual tests, hence we lower the threshold.

In [ ]:
import statsmodels.api as sm
import math

# conduct power analysis to see how many visits we want per variant

# calculate the effect size between A vs. B and B vs. C
effect_a_b = sm.stats.proportion_effectsize(true_ctrs['A'], true_ctrs['B'])
effect_b_c = sm.stats.proportion_effectsize(true_ctrs['B'], true_ctrs['C'])

nind_power = sm.stats.NormalIndPower()
# solve for the sample size for the two effect sizes
sample_a_b = math.ceil(nind_power.solve_power(effect_a_b, alpha=0.05, power=0.8))
sample_b_c = math.ceil(nind_power.solve_power(effect_b_c, alpha=0.05, power=0.8))

print(f'A vs. B Sample Size {sample_a_b}')
print(f'B vs. C Sample Size {sample_b_c}')

A vs. B Sample Size 19
B vs. C Sample Size 119670


In [12]:
# calculate the sample size for the simulation for testing
# use a MDE (minimum detectable effect of 10% and base ctr of 0.21 (around the B and C ctrs))

effect_size = sm.stats.proportion_effectsize(0.21, 0.231)
sample_size = math.ceil(nind_power.solve_power(effect_size, alpha=0.05, power=0.8))
print(f'Sample Size {sample_size}')

Sample Size 6116


In [21]:
from scipy.stats import chi2_contingency
# simulate responses for all three variants

trafficsim = TrafficSimulator(true_ctrs=true_ctrs)
sim_a = trafficsim.simulate_batch('A', 6116)
sim_b = trafficsim.simulate_batch('B', 6116)
sim_c = trafficsim.simulate_batch('C', 6116)

# conduct chi-squared test on all three 
contingency_table = [[np.count_nonzero(sim_a), 6116 - np.count_nonzero(sim_a)], 
                     [np.count_nonzero(sim_b), 6116 - np.count_nonzero(sim_b)],
                     [np.count_nonzero(sim_c), 6116 - np.count_nonzero(sim_c)]]

chi_res = chi2_contingency(contingency_table)

In [24]:
chi_stat, p_value, dof, expected = chi_res
print(f"Chi-square statistic: {chi_stat:.4f}")
print(f"p-value: {p_value}")
print(f"Degrees of freedom: {dof}")
print(f"Expected counts:\n{expected}")
print(f"Actual counts:\n{contingency_table}")

Chi-square statistic: 3637.4339
p-value: 0.0
Degrees of freedom: 2
Expected counts:
[[2225.33333333 3890.66666667]
 [2225.33333333 3890.66666667]
 [2225.33333333 3890.66666667]]
Actual counts:
[[4078, 2038], [1280, 4836], [1318, 4798]]


In [ ]:
# also conduct pairwise test using z-test

a_b_zstat, a_b_pval = sm.stats.proportions_ztest([contingency_table[0][0], contingency_table[1][0]], [6116, 6116])
a_c_zstat, a_c_pval = sm.stats.proportions_ztest([contingency_table[0][0], contingency_table[2][0]], [6116, 6116])
b_c_zstat, b_c_pval = sm.stats.proportions_ztest([contingency_table[1][0], contingency_table[2][0]], [6116, 6116])

# we apply a Bonferroni correction since we are running 3 separate tests
alpha_corrected = 0.05 / 3

print(f"A vs B: z={a_b_zstat:.4f}, p={a_b_pval}, significant={a_b_pval < alpha_corrected}")
print(f"A vs C: z={a_c_zstat:.4f}, p={a_c_pval}, significant={a_c_pval < alpha_corrected}")
print(f"B vs C: z={b_c_zstat:.4f}, p={b_c_pval}, significant={b_c_pval < alpha_corrected}")

A vs B: z=50.9906, p=0.0, significant=True
A vs C: z=50.2598, p=0.0, significant=True
B vs C: z=-0.8401, p=0.4008756869346598, significant=False
